In [1]:
import random
import json
import numpy as np

In [2]:
tasks = ["legal", "FAQ"]
types = ["harmonized", "raw"]
error_types = ["false_negative", "false_positive"]

combinations = [(t, ty, e) for t in tasks for ty in types for e in error_types]
combinations

[('legal', 'harmonized', 'false_negative'),
 ('legal', 'harmonized', 'false_positive'),
 ('legal', 'raw', 'false_negative'),
 ('legal', 'raw', 'false_positive'),
 ('FAQ', 'harmonized', 'false_negative'),
 ('FAQ', 'harmonized', 'false_positive'),
 ('FAQ', 'raw', 'false_negative'),
 ('FAQ', 'raw', 'false_positive')]

In [3]:
def read_data(comb):
    task = comb[0]
    desc_type = comb[1]
    error = comb[2]
    data = []
    with open(f"../results/LLM_as_judge/QueryDocMatch_{error}_all_{task}_{desc_type}.jsonl", "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = {}
for comb in combinations:
    comb_name = "-".join(comb)
    data[comb_name] = read_data(comb)

In [4]:
def read_jsonl(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))

    return data

topic_format_edu = read_jsonl("../data/weborganizer/topic_format_edu.jsonl")

In [5]:
# Print one example doc and its descriptors for the paper
for i, doc in enumerate(topic_format_edu):
    if len(doc["document"]) < 1000 and len(doc["document"]) > 800:
        print(doc["document"])
        best_idx = np.argmax(doc["similarity"])
        print(doc["descriptors"][best_idx])
        print(doc["harmonized_descriptors"])
        break

I am now the proud owner of a beautiful Black (the fastest colour of course) GCSRT8 here in Western Australia
. There is a growing number of these beasts here although I have yet to see one on the road.
You seem to have a fantastic SRT8 community thing going in this forum, I only wish that there was a GCSRT8 group like you here. (If there is an active SRT8 forum/group in Oz, can someone let me have the details?)
I have gathered a lot of useful info from the forum to help me with my next move regarding performance mods and have decided the first will be to change my tstat to a 180
(and maybe the fan mod - although I am not that comfortable in cutting wires etc to get it done)
Can anyone please advise where I can get these thermostats in Australia? What brands are available here? Part numbers? Our local auto chain store (Supa Cheap) hardly even knows what they are let alone if they stock them!
Any help would be appreciated.
["car enthusiast; The document is written by a car enthusiast, i

In [6]:
def print_model_validation_scores(docs):
    no_counter = 0
    partial_counter =0 
    yes_counter = 0
    for doc in docs:
        response = doc["response"]
        if response.lower().endswith("no"):
            no_counter += 1
        elif response.lower().endswith("partial"):
            partial_counter += 1
        elif response.lower().endswith("yes"):
            yes_counter += 1
        else:
            print("Invalid answer:", reponse)

    total_responses = no_counter + partial_counter + yes_counter
    no_relative = round(no_counter/total_responses, 3)
    partial_relative = round(partial_counter/total_responses, 3)
    yes_relative = round(yes_counter/total_responses, 3)
    print(f"NO: {no_counter} ({no_relative})")
    print(f"PARTIAL: {partial_counter} ({partial_relative})")
    print(f"YES: {yes_counter} ({yes_relative})")

In [7]:
for k, v in data.items():
    print(k)    
    print_model_validation_scores(v)
    print()

legal-harmonized-false_negative
NO: 224 (0.683)
PARTIAL: 33 (0.101)
YES: 71 (0.216)

legal-harmonized-false_positive
NO: 376 (0.543)
PARTIAL: 176 (0.254)
YES: 141 (0.203)

legal-raw-false_negative
NO: 232 (0.665)
PARTIAL: 34 (0.097)
YES: 83 (0.238)

legal-raw-false_positive
NO: 249 (0.397)
PARTIAL: 223 (0.356)
YES: 155 (0.247)

FAQ-harmonized-false_negative
NO: 38 (0.11)
PARTIAL: 25 (0.072)
YES: 282 (0.817)

FAQ-harmonized-false_positive
NO: 82 (0.519)
PARTIAL: 24 (0.152)
YES: 52 (0.329)

FAQ-raw-false_negative
NO: 46 (0.144)
PARTIAL: 20 (0.062)
YES: 254 (0.794)

FAQ-raw-false_positive
NO: 103 (0.399)
PARTIAL: 48 (0.186)
YES: 107 (0.415)



In [8]:
def get_matching_descriptors(data_name):
    task, desc_type, error_type = data_name.split("-")
    path = f"../results/LLM_as_judge/QueryDescriptorMatch_{desc_type.lower()}_{task.lower()}.jsonl"
    matching_descriptors = read_jsonl(path)
    descriptors = []
    for resp in matching_descriptors:
        if resp["response"].lower().strip().endswith("yes"):
            descriptors.append(resp["example"]["descriptor"])

    return descriptors

def print_sample(docs, matching_descriptors, model_judgements=None, sample_size=5):
    all_docs = {
        "no_docs": [],
        "partial_docs": [],
        "yes_docs": []
    }
    for doc in docs:        
        doc_text = doc["example"]["document"]
        for text in topic_format_edu:
            if doc_text.strip() == text["document"].strip():
                best_idx = np.argmax(text["similarity"])
                descriptors = text["descriptors"][best_idx]
                harmonized = text["harmonized_descriptors"]
                break
            else:
                descriptors = None
                harmonized = None
        response = doc["response"]
        if response.lower().endswith("no"):
            all_docs["no_docs"].append((doc["example"]["document"], response, descriptors, harmonized))
        elif response.lower().endswith("partial"):
            all_docs["partial_docs"].append((doc["example"]["document"], response, descriptors, harmonized))
        elif response.lower().endswith("yes"):
            all_docs["yes_docs"].append((doc["example"]["document"], response, descriptors, harmonized))
        else:
            print("Invalid answer:", reponse)

    names_to_print = {"no_docs": "LLM decision: NO", "partial_docs": "LLM decision: Partial", "yes_docs": "LLM decision: Yes"}
    random.seed(42)
    for k, v in all_docs.items():
        if model_judgements:
            if "yes" not in model_judgements and k == "yes_docs":
                continue
            if "no" not in model_judgements and k == "no_docs":
                continue
            if "partial" not in model_judgements and k == "partial_docs":
                continue
        print(names_to_print[k])
        random.shuffle(v)
        for i in range(sample_size):
            print(f"++++++++++{i}+++++++++++")
            print("TEXT:", v[i][0])
            print("-------------------------")
            print("EXPLANATION:", v[i][1])
            matching = []
            for desc in v[i][2]:
                if desc in matching_descriptors:
                    matching.append(desc)
            for desc in v[i][3]:
                if desc in matching_descriptors:
                    matching.append(desc)
            if matching:
                print("Matching descriptors:")
                print(matching)
            else:
                print("DESCRIPTORS:", v[i][2])
                print("HARMONIZED:", v[i][3])
            print()
        print("==================")

In [10]:
tasks = ["legal"]
types = ["raw"]
error_types = ["false_negative"]

data_to_print = ["-".join((t, ty, e)) for t in tasks for ty in types for e in error_types]
model_judgements_to_print = ["yes"]
for k, v in data.items():    
    if k in data_to_print:
        matching_descriptors = get_matching_descriptors(k)
        print(k)
        print_sample(v, matching_descriptors, model_judgements=model_judgements_to_print)
        print()

legal-raw-false_negative
LLM decision: Yes
++++++++++0+++++++++++
TEXT: The undersigned, by virtue of the authority vested in them, have
concluded the following Agreement.
PURPOSE OF THE AGREEMENT
This Agreement shall govern the exchange of International Busin-
ess Reply Service (IBRS) items between The Postal Administration
of Taiwan, Republic of China and The Public Postal Operator in
Sweden (Sweden Post Ltd. - Company registration number 556451-41
48) including any areas of which the postal administration of t-
hese two countries exercise IBRS responsibilities
As used in this Agreement, the following terms shall have the i-
1.Administration - an abbreviated form used to refer to either
the Postal Administration of Taiwan, Republic of China or the
Public Postal Operator in Sweden (Sweden Post Ltd. - Company
registration number 556451-4148).
2.Articles and sections - articles and sections of this Agreeme-
nt, except when the context indicates an article which is or
can be inserted int

In [30]:
def read_data(desc_type):
    data = []
    with open(f"../results/LLM_as_judge/QueryDocMatch_validated_all_sarcasm_{desc_type}.jsonl", "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

sarcasm = {
    "sarcasm_harm": read_data("harmonized"),
    "sarcasm_raw": read_data("raw"),
}

In [31]:
for k, v in sarcasm.items():    
    print(k)
    print_sample(v)
    print()

sarcasm_harm
LLM decision: NO
++++++++++0+++++++++++
TEXT: In This Post, You Are Going to Get The Best Betron YSM1000 Headphones Black Friday Deals 2021.
The bass heads of this world listen up. If you are looking for low-cost earbuds with overwhelming bass, you must read this.
Betron YSM1000 Headphones Black Friday Deals
The Betron YSM1000 are wired in-ear earphones with a “blingy” design as well as deep bass. These are a strong alternative to Beats (but much cheaper).
Everyone enjoys experiencing the adventure of surround audio at the movie theater. You can really feel the audio of a vehicle taking off or an animal thumping through the forest. Does not it send a chill down your spine? If you desire a similar kind of experience when listening to songs so you can really feel every beat of the drum and also every string of the guitar, what you need is bass-boost earphones.
If you are considering acquiring the Betron YSM1000 earphones, here’s our evaluation to aid you to make your choice.